[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/llm_3d.ipynb)

# Word geometry in three dimensions

Ten pictures a mouse can turn, on the model the lecture fits: 138 types of
English, each one a point, and the three directions that spread them furthest
out of the hundred the model holds. Run each cell with **Shift and Enter**, and
drag inside a picture to turn it.

Three coordinates out of a hundred move every angle they are asked about, so
each picture prints the comparison the model makes in all hundred beside the
one on the screen. The gap between the two is what a projection costs.

In [ ]:
import json
import sys

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = ("colab" if "google.colab" in sys.modules
                         else "notebook_connected")

INK, AMBER, GREY, BAND = "#1e3a5f", "#b45309", "#64748b", "#dfe4ec"
# The four groups in the colours pages 33 and 57 of the deck give them.
SHADE = {"animals": "#236627", "computing": GREY, "royalty": AMBER,
         "money": INK, "rest": BAND}


def space(title, height=560, legend=True):
    """A cube with three named directions and room for a title."""
    fig = go.Figure()
    fig.update_layout(
        title=dict(text=title, x=0.02, font=dict(size=15, color=INK)),
        height=height, margin=dict(l=0, r=0, t=44, b=0),
        showlegend=legend, template="plotly_white",
        scene=dict(xaxis_title="direction 1", yaxis_title="direction 2",
                   zaxis_title="direction 3", aspectmode="data"))
    return fig


def cloud(fig, points, labels, colour, name, size=4, opacity=1.0):
    """One group of words as points a mouse can turn."""
    points = np.atleast_2d(points)
    fig.add_trace(go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2], mode="markers",
        marker=dict(size=size, color=colour, opacity=opacity),
        text=labels, hovertemplate="%{text}<extra></extra>", name=name))


def segment(fig, a, b, colour=GREY, width=3, dash="solid", name=""):
    """A line from one word to another."""
    fig.add_trace(go.Scatter3d(
        x=[a[0], b[0]], y=[a[1], b[1]], z=[a[2], b[2]], mode="lines",
        line=dict(color=colour, width=width, dash=dash), name=name,
        showlegend=bool(name), hoverinfo="skip"))


def tag(fig, points, labels, colour=INK, size=11):
    """The word printed beside its own point."""
    points = np.atleast_2d(points)
    fig.add_trace(go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2], mode="text",
        text=labels, textfont=dict(size=size, color=colour),
        hoverinfo="skip", showlegend=False))


def room(fig, points, pad=0.22):
    """Ranges wide enough that a word printed at the edge stays on the page."""
    points = np.atleast_2d(points)
    span = {}
    for k, axis in enumerate("xyz"):
        low, high = float(points[:, k].min()), float(points[:, k].max())
        step = pad * (high - low or 1.0)
        span[axis + "axis"] = dict(range=[low - step, high + step])
    fig.update_layout(scene=span)


def bar3d(fig, x, y, height, colour, width=0.34, name=""):
    """One upright box, because attention is read token by token."""
    xs = [x - width, x + width]
    ys = [y - width, y + width]
    corners = [(a, b, c) for c in (0.0, height) for b in ys for a in xs]
    faces = [(0, 1, 2), (1, 2, 3), (4, 5, 6), (5, 6, 7), (0, 1, 4), (1, 4, 5),
             (2, 3, 6), (3, 6, 7), (0, 2, 4), (2, 4, 6), (1, 3, 5), (3, 5, 7)]
    fig.add_trace(go.Mesh3d(
        x=[c[0] for c in corners], y=[c[1] for c in corners],
        z=[c[2] for c in corners], i=[f[0] for f in faces],
        j=[f[1] for f in faces], k=[f[2] for f in faces], color=colour,
        opacity=1.0, flatshading=True, name=name, showlegend=False,
        hovertemplate="%%{z:.3f}<extra>%s</extra>" % name))


def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

## The words and their coordinates

The types, the group each one belongs to, and the three coordinates, read off
the model fitted on 17 million tokens of Wikipedia text.

In [ ]:
D = json.loads(r"""
{"analogy":{"rank":1,"score":0.821,"target":"berlin","top":[["berlin",
0.821],["munich",0.783],["vienna",0.744],["leipzig",0.725],["frankfurt",
0.708]]},"attention":{"finance":{"cosines":[0.112,0.228,0.439,0.354,0.082,
0.092],"focus":"bank","keys":["the","raised","interest","rates","in",
"june"],"moved":{"credit":0.681,"money":0.678,"river":0.215,"water":0.133},
"sentence":["the","bank","raised","interest","rates","in","june"],
"weights":[0.077,0.137,0.394,0.257,0.066,0.069]},"river":{"cosines":[0.112,
0.224,-0.005,0.12,0.082,0.016],"focus":"bank","keys":["the","river","was",
"covered","in","ice"],"moved":{"credit":0.504,"money":0.47,"river":0.532,
"water":0.271},"sentence":["the","river","bank","was","covered","in",
"ice"],"weights":[0.172,0.3,0.096,0.179,0.148,0.106]}},
"corpus":{"coverage":0.911,"dims":100,"hapax":118519,"tokens":17005207,
"types":253854,"vocab":12000,"window":5},"group":["computing","rest",
"rest","rest","money","rest","rest","rest","rest","animals","rest","rest",
"rest","animals","rest","rest","rest","computing","rest","rest","rest",
"rest","animals","money","royalty","rest","money","rest","computing",
"rest","rest","rest","rest","rest","rest","rest","animals","rest","rest",
"rest","rest","royalty","rest","rest","rest","money","animals","rest",
"rest","rest","rest","rest","rest","rest","rest","rest","rest","rest",
"computing","rest","animals","rest","rest","rest","rest","rest","rest",
"rest","rest","rest","rest","rest","rest","royalty","rest","rest","rest",
"rest","rest","money","rest","rest","computing","rest","rest","rest",
"rest","rest","rest","rest","money","rest","rest","rest","rest","rest",
"rest","rest","rest","rest","rest","rest","rest","royalty","rest",
"royalty","rest","rest","rest","rest","rest","rest","rest","rest","rest",
"rest","rest","computing","rest","rest","rest","rest","rest","rest","rest",
"rest","royalty","rest","rest","rest","rest","rest","rest","rest","rest",
"rest","rest","rest"],"money":["fund","banks","monetary","imf","financial",
"exchange","money","credit","loan","currency"],
"neighbours":{"bank":[["fund",0.762],["banks",0.732],["monetary",0.722],
["imf",0.686],["financial",0.668],["exchange",0.654]],"cat":[["dog",0.809],
["dogs",0.715],["bird",0.676],["cats",0.674],["breed",0.674],["frog",
0.659]],"computer":[["computers",0.853],["computing",0.801],["hardware",
0.79],["machines",0.772],["graphics",0.738],["systems",0.737]],
"king":[["iii",0.892],["crowned",0.884],["iv",0.88],["ii",0.87],["vii",
0.844],["throne",0.826]]},"nextword":{"prefix":["the","united"],
"seen":10812,"top":[["states",7910,0.732],["kingdom",1967,0.182],
["nations",634,0.059],["arab",56,0.005],["provinces",39,0.004],
["methodist",28,0.003]]},"pairs":{"apple|computer":0.616,
"apple|fruit":0.197,"bank|money":0.549,"bank|river":0.224,
"cat|chair":0.142,"cat|computer":0.109,"cat|dog":0.809,"king|queen":0.714},
"share":0.257,"water":["river","water","shore","flood","ice","liquid",
"vapor"],"words":["algorithm","amiga","apple","bad","bank","banks",
"berlin","better","big","bird","boiling","breed","buy","cat","cats",
"chair","cold","computer","computers","computing","consort","covered",
"cow","credit","crown","crowned","currency","dancer","data","daughter",
"day","decrease","des","difficult","dioxide","doctor","dog","dogs","doing",
"dos","du","emperor","engineer","everything","exchange","financial","fish",
"flood","florence","france","frankfurt","frog","fruit","fund","germany",
"good","graphics","hard","hardware","he","horse","hot","ibm","ice","ii",
"iii","imf","in","increase","interest","italy","iv","june","king","la",
"le","leipzig","likely","liquid","loan","loans","lyon","machine",
"machines","macintosh","man","microsoft","milan","mixture","monetary",
"money","moscow","munich","night","north","nothing","nurse","of","oxygen",
"paris","pc","peace","philosopher","prince","programmer","queen","quickly",
"raised","rates","river","rome","scientist","sell","she","shore","singer",
"small","software","soldier","something","south","sur","systems","teacher",
"the","thing","throne","too","vapor","venice","vienna","vii","war","was",
"water","woman","worse","your"],"xyz":[[0.1516,-0.0619,-0.019],[0.4252,
-0.6049,-0.1366],[0.4076,-0.5786,-0.135],[0.253,0.389,-0.2552],[0.1088,
0.0709,0.6851],[0.2218,0.1091,0.6361],[-0.2932,0.0166,0.125],[0.326,0.1992,
-0.1904],[0.1668,0.2308,-0.1875],[0.1149,0.3197,-0.224],[0.0997,0.2309,
0.0703],[0.1653,0.3484,-0.2785],[0.2783,0.1676,0.2513],[0.142,0.3353,
-0.3346],[0.1916,0.3308,-0.2746],[-0.1172,0.002,0.0089],[0.0719,0.2404,
-0.0596],[0.4304,-0.645,-0.1427],[0.4759,-0.6351,-0.191],[0.3622,-0.5227,
-0.0769],[-0.5524,-0.1516,-0.124],[0.0912,0.195,-0.0992],[0.1337,0.2849,
-0.3371],[0.2294,0.1222,0.6175],[-0.421,-0.0829,-0.0162],[-0.6833,-0.1922,
-0.1332],[0.1544,0.0643,0.6044],[-0.0678,0.0606,-0.0825],[0.211,-0.2644,
0.0755],[-0.4782,-0.094,-0.1748],[-0.1175,-0.0242,0.0694],[0.1159,0.1731,
0.2728],[-0.1847,-0.0473,0.0989],[0.2691,0.2159,-0.2045],[0.0767,0.1089,
0.1213],[-0.0553,0.1079,-0.1374],[0.152,0.4197,-0.3488],[0.1777,0.3794,
-0.3551],[0.2759,0.3058,-0.2519],[0.3734,-0.6368,-0.1577],[-0.2054,-0.0161,
0.0108],[-0.6126,-0.1844,-0.1188],[0.0278,-0.0835,-0.0673],[0.2172,0.3294,
-0.3161],[0.1659,0.0426,0.6538],[0.2008,0.0521,0.6366],[0.1245,0.2491,
-0.1583],[0.1682,0.0212,0.1782],[-0.4255,-0.1181,0.0718],[-0.5344,-0.1909,
0.0852],[-0.3228,-0.0564,0.1843],[0.0344,0.2269,-0.2478],[0.0911,0.2091,
-0.1488],[0.1522,0.0835,0.6919],[-0.3366,-0.0741,0.1797],[0.2161,0.4569,
-0.283],[0.3528,-0.5789,-0.1245],[0.3995,0.0607,-0.1984],[0.4818,-0.6418,
-0.1237],[-0.0812,0.1074,-0.11],[0.0911,0.3153,-0.294],[0.1038,0.2606,
-0.127],[0.4377,-0.6483,-0.0999],[0.049,0.1957,-0.0274],[-0.62,-0.3088,
-0.1331],[-0.5964,-0.2514,-0.1581],[0.1186,0.0782,0.6396],[-0.1622,-0.084,
0.0431],[0.1807,0.1335,0.3851],[0.1785,0.2287,0.435],[-0.4181,-0.1263,
0.1364],[-0.6265,-0.2055,-0.123],[-0.1717,-0.0878,0.061],[-0.6625,-0.1907,
-0.1285],[-0.259,-0.0484,0.013],[-0.2641,-0.0474,-0.0037],[-0.308,0.0238,
0.1039],[0.1774,0.2279,-0.1679],[0.1132,0.1745,0.0599],[0.1727,0.0927,
0.628],[0.172,0.0737,0.6977],[-0.3401,-0.1478,0.0521],[0.345,-0.366,
-0.1558],[0.4487,-0.5279,-0.1729],[0.3885,-0.6849,-0.1618],[0.0293,0.354,
-0.3623],[0.3937,-0.5942,-0.0819],[-0.4873,-0.1521,0.0892],[0.0793,0.2055,
0.0249],[0.1186,0.0696,0.7331],[0.2565,0.2282,0.5264],[-0.2639,-0.0649,
0.0915],[-0.3155,0.0305,0.1443],[-0.0225,0.2069,-0.1811],[-0.062,0.0088,
0.0651],[0.1915,0.3948,-0.2631],[-0.1098,0.0724,-0.0581],[-0.2274,-0.1013,
0.0063],[0.0827,0.1634,0.1363],[-0.3963,-0.054,0.0793],[0.4216,-0.638,
-0.16],[-0.1389,0.0747,0.1593],[-0.1566,-0.0161,0.0008],[-0.6301,-0.1355,
-0.1484],[0.2952,-0.247,-0.1281],[-0.5249,-0.1031,-0.1653],[0.2455,0.1032,
-0.0735],[0.0056,0.1641,-0.0173],[0.193,0.0926,0.5091],[0.0116,0.0617,
0.0839],[-0.4347,-0.1252,0.0238],[-0.035,0.0052,-0.0659],[0.2798,0.0267,
0.1704],[-0.0788,0.1733,-0.226],[0.0161,0.0905,0.0639],[-0.0498,0.0545,
-0.0378],[0.2168,0.1528,-0.0421],[0.4058,-0.5788,-0.1329],[-0.2482,0.0848,
-0.2047],[0.2234,0.4123,-0.315],[-0.0533,-0.0065,0.0616],[-0.2565,-0.0602,
0.0696],[0.397,-0.5758,-0.1219],[-0.1217,0.1292,-0.0875],[-0.1094,-0.0472,
0.0329],[0.1694,0.3935,-0.3314],[-0.5561,-0.1313,-0.1464],[0.259,0.3951,
-0.2538],[0.0844,0.155,0.0994],[-0.4961,-0.1246,0.1356],[-0.4133,-0.07,
0.1021],[-0.5884,-0.2193,-0.1272],[-0.1699,-0.0642,0.0585],[-0.1965,
-0.0779,-0.1121],[0.1159,0.2382,0.1051],[-0.0794,0.2096,-0.2859],[0.1596,
0.2099,-0.1421],[0.2258,0.3661,-0.3064]]}
""")
WORDS = D["words"]
XYZ = np.array(D["xyz"])
AT = {w: i for i, w in enumerate(WORDS)}
GROUP = dict(zip(WORDS, D["group"]))


def at(word):
    """The three coordinates of one word."""
    return XYZ[AT[word]]


print("%d types, %d coordinates drawn out of %d fitted"
      % (len(WORDS), 3, D["corpus"]["dims"]))
print("the three directions carry %.1f%% of the spread among them"
      % (100 * D["share"]))

## The cloud

Four groups of six were chosen before the model was fitted: animals, computing,
royalty and money. Nothing told the model which word belongs to which group.
Turn the cloud until the four separate.

In [ ]:
fig = space("Four groups of six, and the types around them")
rest = [w for w in WORDS if GROUP[w] == "rest"]
cloud(fig, XYZ[[AT[w] for w in rest]], rest, SHADE["rest"], "other types",
      size=3, opacity=0.55)
for name in ("animals", "computing", "royalty", "money"):
    members = [w for w in WORDS if GROUP[w] == name]
    cloud(fig, XYZ[[AT[w] for w in members]], members, SHADE[name], name,
          size=6)
    tag(fig, XYZ[[AT[w] for w in members]], members, SHADE[name], size=10)
fig.show()

## The sphere

Dividing each point by its own length puts every type on a sphere of radius
one, where the only quantity left between two types is the angle. Cosine
similarity is that angle, and it runs from 1 for two types pointing the same
way to 0 for two at a right angle. On a sphere of three coordinates the types
crowd together, and the second line printed under it is what the same two
types score in all hundred.

In [ ]:
directions = XYZ / np.linalg.norm(XYZ, axis=1, keepdims=True)

u, v = np.mgrid[0:2 * np.pi:60j, 0:np.pi:30j]
fig = space("Every length divided out, so only the angle is left")
fig.add_trace(go.Surface(
    x=np.cos(u) * np.sin(v), y=np.sin(u) * np.sin(v), z=np.cos(v),
    colorscale=[[0, BAND], [1, BAND]], opacity=0.28, showscale=False,
    hoverinfo="skip", name="the unit sphere"))
for name in ("animals", "computing", "royalty", "money"):
    members = [w for w in WORDS if GROUP[w] == name]
    cloud(fig, directions[[AT[w] for w in members]], members, SHADE[name],
          name, size=6)
fig.show()

print("on the sphere drawn here: cat and dog %.3f, cat and chair %.3f"
      % (cosine(at("cat"), at("dog")), cosine(at("cat"), at("chair"))))
print("in all hundred coordinates: cat and dog %.3f, cat and chair %.3f"
      % (D["pairs"]["cat|dog"], D["pairs"]["cat|chair"]))

## Two angles

`cat` and `dog` stand close together, `chair` swings away. Both angles are
wider in the hundred coordinates the model holds than the three drawn here
show, and the two printed numbers are the same comparison in the two
spaces.

In [ ]:
fig = space("One angle wide, one angle narrow", legend=False)
origin = np.zeros(3)
for word, colour in (("cat", INK), ("dog", INK), ("chair", AMBER)):
    point = at(word) / np.linalg.norm(at(word))
    segment(fig, origin, point, colour, width=6)
    tag(fig, point * 1.08, [word], colour)
cloud(fig, origin, ["origin"], GREY, "origin", size=4)
room(fig, np.vstack([origin] + [at(w) / np.linalg.norm(at(w))
                                for w in ("cat", "dog", "chair")]))
fig.show()

for a, b in (("cat", "dog"), ("cat", "chair")):
    print("%s and %s: %.3f in the three drawn, %.3f in the hundred fitted"
          % (a, b, cosine(at(a), at(b)), D["pairs"]["%s|%s" % (a, b)]))
print("three coordinates hold %.0f%% of the spread, and the rest of the "
      "angle is in the ninety seven left out" % (100 * D["share"]))

## Nearest neighbours

The six types nearest `cat` in all hundred coordinates, each joined to it by a
line as thick as the score. Change the word in the first line to `bank`,
`computer` or `king`.

In [ ]:
query = "cat"        # try "bank", "computer" or "king"

near = D["neighbours"][query]
fig = space("The six nearest types of %s, and the rest of the cache" % query)
others = [w for w in WORDS if w != query and w not in [n for n, _ in near]]
cloud(fig, XYZ[[AT[w] for w in others]], others, SHADE["rest"], "other types",
      size=3, opacity=0.5)
cloud(fig, at(query), [query], AMBER, query, size=9)
tag(fig, at(query) * 1.0 + np.array([0, 0, 0.06]), [query], AMBER)
for word, score in near:
    if word in AT:
        segment(fig, at(query), at(word), GREY, width=2 + 8 * (score - 0.6))
        cloud(fig, at(word), [word], INK, word, size=6)
        tag(fig, at(word), ["%s %.3f" % (word, score)], INK, size=10)
room(fig, XYZ[[AT[w] for w, _ in near if w in AT] + [AT[query]]], pad=0.6)
fig.show()

print("nearest six in all hundred coordinates:",
      ", ".join("%s %.3f" % (w, s) for w, s in near))

## The parallelogram

The displacement from `france` to `paris` carried over to `germany` lands
beside `berlin`. A projection is linear, so the parallelogram you see in three
coordinates is the parallelogram the model holds in a hundred, and the gap
between the constructed corner and `berlin` is the part of the error that
survives the projection.

In [ ]:
made = at("paris") - at("france") + at("germany")

fig = space("paris minus france plus germany, and where berlin sits",
            legend=False)
for word in ("france", "paris", "germany", "berlin"):
    cloud(fig, at(word), [word], INK, word, size=7)
    tag(fig, at(word), [word], INK)
segment(fig, at("france"), at("paris"), GREY, width=5)
segment(fig, at("germany"), made, GREY, width=5)
segment(fig, at("france"), at("germany"), GREY, width=2, dash="dash")
segment(fig, at("paris"), made, GREY, width=2, dash="dash")
cloud(fig, made, ["constructed"], AMBER, "constructed", size=9)
tag(fig, made, ["constructed"], AMBER)
segment(fig, made, at("berlin"), AMBER, width=7, dash="dot")
room(fig, np.vstack([XYZ[[AT[w] for w in
                          ("france", "paris", "germany", "berlin")]], made]))
fig.show()

print("the constructed point and berlin, in all hundred: %.3f, rank %d"
      % (D["analogy"]["score"], D["analogy"]["rank"]))
print("what follows it:",
      ", ".join("%s %.3f" % (w, s) for w, s in D["analogy"]["top"][1:]))

## Turned axes

Rotating the coordinate system moves every number in the table and leaves every
angle where it was. The two clouds below are the same words before and after a
turn of the axes, and the printed cosines agree to six decimals. No single
direction is the animal direction, because a turn can put it anywhere.

In [ ]:
angle = 1.1          # turn the axes by this many radians and look again

axis = np.array([1.0, 2.0, 3.0])
axis = axis / np.linalg.norm(axis)
cross = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
Q = (np.eye(3) + np.sin(angle) * cross
     + (1 - np.cos(angle)) * cross @ cross)
turned = XYZ @ Q.T

fig = space("The same words, and the same words on turned axes")
for name in ("animals", "computing", "royalty", "money"):
    members = [w for w in WORDS if GROUP[w] == name]
    cloud(fig, XYZ[[AT[w] for w in members]], members, SHADE[name], name,
          size=6)
    cloud(fig, turned[[AT[w] for w in members]], members, SHADE[name],
          name + ", turned", size=6, opacity=0.35)
fig.show()

for a, b in (("cat", "dog"), ("king", "queen"), ("cat", "chair")):
    here = cosine(at(a), at(b))
    there = cosine(turned[AT[a]], turned[AT[b]])
    print("%s and %s: %.6f before the turn, %.6f after" % (a, b, here, there))

## Two neighbourhoods, one point

The cosines printed below are read off the three coordinates drawn. `bank`
occurs beside money and beside water, and the model fits one point for
the type. That point sits between the two neighbourhoods and belongs to
neither. Contextual models answer this by giving the token a position that
moves with the sentence.

In [ ]:
money, water = D["money"], D["water"]
fig = space("One point for bank, and the two neighbourhoods around it")
cloud(fig, XYZ[[AT[w] for w in money]], money, SHADE["money"], "money", size=6)
tag(fig, XYZ[[AT[w] for w in money]], money, SHADE["money"], size=10)
cloud(fig, XYZ[[AT[w] for w in water]], water, "#0f766e", "water", size=6)
tag(fig, XYZ[[AT[w] for w in water]], water, "#0f766e", size=10)
cloud(fig, at("bank"), ["bank"], AMBER, "bank", size=11)
tag(fig, at("bank") + np.array([0, 0, 0.07]), ["bank"], AMBER, size=13)

for group, colour in ((money, SHADE["money"]), (water, "#0f766e")):
    middle = XYZ[[AT[w] for w in group]].mean(0)
    segment(fig, at("bank"), middle, colour, width=5)
    print("bank and the middle of that neighbourhood, in the three drawn: "
          "%.3f" % cosine(at("bank"), middle))
room(fig, XYZ[[AT[w] for w in money + water + ["bank"]]])
fig.show()

## The simplex

After `the united`, the corpus puts three words at the head of the
distribution, and the three are renormalised here to add to one. Three probabilities that add to one are a point on a triangle,
and temperature slides that point along the curve: low temperature to the
corner the corpus favours, high temperature to the middle, where the three
become interchangeable.

In [ ]:
temperature = 1.0    # 0.2 sharpens the draw, 3.0 flattens it

top = D["nextword"]["top"][:3]
names = [w for w, _, _ in top]
counts = np.array([c for _, c, _ in top], dtype=float)
logit = np.log(counts / counts.sum())


def softmax(z, t):
    e = np.exp((z - z.max()) / t)
    return e / e.sum()


path = np.array([softmax(logit, t) for t in np.linspace(0.2, 3.0, 60)])
corners = np.eye(3)
fig = space("Three words the corpus puts after the united, and the draw",
            legend=False)
fig.update_layout(scene=dict(xaxis_title="", yaxis_title="", zaxis_title="",
                             aspectmode="cube"))
fig.add_trace(go.Mesh3d(x=corners[:, 0], y=corners[:, 1], z=corners[:, 2],
                        i=[0], j=[1], k=[2], color=BAND, opacity=0.35,
                        hoverinfo="skip"))
fig.add_trace(go.Scatter3d(x=path[:, 0], y=path[:, 1], z=path[:, 2],
                           mode="lines", line=dict(color=GREY, width=5),
                           hoverinfo="skip"))
here = softmax(logit, temperature)
cloud(fig, here, ["temperature %.1f" % temperature], AMBER, "draw", size=9)
tag(fig, corners, names, INK)
fig.show()

print("the corpus, over the three words at the head of the list:", ", ".join("%s %.3f" % (w, p) for w, _, p in top))
print("at temperature %.1f:" % temperature,
      ", ".join("%s %.3f" % (w, p) for w, p in zip(names, here)))

## What one word reads

Attention gives every token a share of every other token. The bars are the
shares the fitted model gives `bank` in two sentences, and the tallest one in
each is what the sense turns on: `interest` in the money sentence and `river`
in the other. The point that comes out of the step is printed under the
picture, and it moves with the sentence.

In [ ]:
finance, river = D["attention"]["finance"], D["attention"]["river"]

fig = go.Figure()
sentences = ["the bank raised interest rates in june",
             "the river bank was covered in ice"]
for row, reading in enumerate((finance, river)):
    hottest = int(np.argmax(reading["weights"]))
    for k, (word, share) in enumerate(zip(reading["keys"],
                                          reading["weights"])):
        bar3d(fig, k, row * 2.0, share, AMBER if k == hottest else INK,
              name="%s reads %s" % (reading["focus"], word))
        tag(fig, [[k, row * 2.0, share + 0.05]], [word], GREY, size=10)

fig.update_layout(
    title=dict(text="What bank reads in two sentences", x=0.02,
               font=dict(size=15, color=INK)),
    height=560, margin=dict(l=0, r=0, t=44, b=0), template="plotly_white",
    showlegend=False,
    scene=dict(xaxis=dict(title="", showticklabels=False, range=[-1, 6]),
               yaxis=dict(title="", tickvals=[0, 2], ticktext=sentences,
                          range=[-1.2, 3.2]),
               zaxis=dict(title="weight", range=[0, 0.5]),
               camera=dict(eye=dict(x=1.5, y=-1.6, z=0.85)),
               aspectratio=dict(x=1.7, y=1.2, z=0.75)))
fig.show()

for reading, name in ((finance, "money sentence"), (river, "river sentence")):
    print("after the %s, the point for bank sits at" % name,
          ", ".join("%s %.3f" % (w, s)
                    for w, s in sorted(reading["moved"].items(),
                                       key=lambda pair: -pair[1])))

## The cost of a longer context

Every token reads every token, so the attention step grows with the square of
the context length while the rest of the network grows with the length. The
surface is the count of multiply adds, and the axis is a logarithm because the
range spans eight decades.

In [ ]:
width = 768          # coordinates per token, 4096 in a large model

length = np.logspace(2, 5.2, 60)
layers = np.linspace(6, 96, 60)
L, H = np.meshgrid(length, layers)
work = np.log10(H * L ** 2 * width)

fig = go.Figure(go.Surface(
    x=np.log10(L), y=H, z=work, colorscale=[[0, BAND], [1, AMBER]],
    showscale=True,
    hovertemplate=("context 10^%{x:.1f} tokens<br>%{y:.0f} layers"
                   "<br>10^%{z:.1f} multiply adds<extra></extra>")))
fig.update_layout(
    title=dict(text="What a longer context costs the attention step", x=0.02,
               font=dict(size=15, color=INK)),
    height=560, margin=dict(l=0, r=0, t=44, b=0), template="plotly_white",
    scene=dict(xaxis_title="log10 context length",
               yaxis_title="layers", zaxis_title="log10 multiply adds"))
fig.show()

print("4,000 tokens to 128,000 tokens multiplies the attention step by %d"
      % ((128000 / 4000) ** 2))

## What you can say now

- A word is a point, a hundred numbers long, and the three directions drawn
  here are the three that spread these 138 types furthest apart.
- Meaning sits in the angle between two points. Dividing out the lengths puts
  every type on one sphere, where the angle is all there is.
- The axes carry nothing: turning them rewrites every coordinate and leaves
  every angle unchanged.
- One point for one type is the limit under all of it, which `bank` shows by
  standing between money and water.
- Attention and temperature act on this same geometry, one by mixing points and
  one by sharpening the draw.

The deck **Large Language Models** carries the measurements these pictures
turn.